# L01 · Probability, Gradients, and PyTorch Survival Kit

## Goal

- compute log-probability, entropy, and KL
- explain an expectation gradient
- inspect gradient flow after detach

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L01:toy:42").hexdigest()
print(f"lesson=L01 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L01 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:30408c6216764a44279ea7141e1fd584ff1b221be0556e06c7bbed32db722e95 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: whole map → **probability and gradients** → policy gradient

$$H(p)=-\sum_i p_i\log p_i,\qquad D_{KL}(p\|q)=\sum_i p_i\log\frac{p_i}{q_i}$$

Log-probability turns products into sums and exposes sensitivity of a chosen action. Entropy measures spread; KL is a directional discrepancy between distributions. `detach` preserves a value while removing gradient ownership along that path.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** When maximizing entropy and reducing forward KL, what sign do you expect on the largest logit's gradient? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>The terms compete, so inspect autograd rather than guessing. For this input the second-logit gradient is negative.</details>

In [2]:
from rl_study.math import categorical_entropy, categorical_kl
logits = torch.tensor([[0.0, 1.0, -1.0]], requires_grad=True)
other = torch.tensor([[0.4, 0.2, -0.3]])
entropy = categorical_entropy(logits)
forward_kl = categorical_kl(logits, other)
objective = entropy.mean() - forward_kl.mean()
objective.backward()
print({"entropy": round(float(entropy.detach()), 4),
       "kl_p_q": round(float(forward_kl.detach()), 4),
       "gradient": [round(x, 4) for x in logits.grad[0].tolist()]})

{'entropy': 0.8324, 'kl_p_q': 0.2032, 'gradient': [0.3295, -0.5678, 0.2383]}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Package functions based on `log_softmax` provide normalization and numerical stability without manual probability clamping. Reverse KL has different mode-seeking behavior and is not interchangeable.

**Common trap:** Calling `float(tensor)` on a gradient-tracking tensor warns. Detach observation-only values before scalar conversion, but never detach the training loss. Regression tests: `test_probability_matches_torch`, `test_reinforce_sign`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert float(forward_kl.detach()) >= 0.0 and torch.isfinite(logits.grad).all()
print("checks=passed")

checks=passed


**Recall:** Why does swapping `D_KL(p||q)` and `D_KL(q||p)` change the regularizer? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** Entropy and KL were finite, and the three logit gradients sum to approximately zero, matching softmax invariance to a common logit shift.
- Executable checks: `test_probability_matches_torch`, `test_reinforce_sign`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L02 extends one categorical choice to token-sequence log-probabilities and masks.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`